In [1]:
from rag_public_reports.ingest import ingest_pdf
from rag_public_reports.vectorstore import get_vector_store, add_documents, is_already_ingested, search
from rag_public_reports.utils import list_ingested_reports, print_chunk_sample
from rag_public_reports.rag import answer

In [ ]:
vs = get_vector_store()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from rag_public_reports.config import DATA_DIR

# Prends n'importe quel rapport CDC déjà dans data/raw/
pages = PyPDFLoader(str(list((DATA_DIR / "raw").glob("*.pdf"))[0])).load()

# Inspecte 3 pages au hasard
for page in pages[5:8]:
    print(f"\n{'='*50} PAGE {page.metadata.get('page')} {'='*50}")
    print(repr(page.page_content[:500]))  # repr() pour voir les \n explicitement

In [ ]:
from rag_public_reports.vectorstore import get_vector_store, search

vs = get_vector_store()
query = "Quelles sont les recommandations sur la gestion des ressources humaines ?"
docs = search(vs, query, k=10)

for doc in docs:
    m = doc.metadata
    print(f"[{m.get('institution')} {m.get('year')}] section: {m.get('section', '?')[:60]}")
    print(doc.page_content[:200])
    print("---")

In [ ]:
import inspect
from rag_public_reports.ingest import _detect_section_title
print(inspect.getsource(_detect_section_title))

In [6]:
# Reproduis exactement ce que voit _detect_section_title
from langchain_community.document_loaders import PyPDFLoader
from rag_public_reports.config import DATA_DIR
from rag_public_reports.ingest import _detect_section_title, _get_patterns
import re

# Charge le rapport CDC 2024
pdf_path = list((DATA_DIR / "raw").glob("*2024*ccomptes*"))[0]  # adapte si besoin
pages = PyPDFLoader(str(pdf_path)).load()

patterns = _get_patterns("Cour des comptes")

# Inspecte les pages autour de la page 41
for page in pages[38:44]:
    print(f"\n{'='*50} PAGE {page.metadata.get('page')} {'='*50}")
    print(repr(page.page_content[:300]))
    detected = _detect_section_title(page.page_content, patterns)
    print(f"→ Titre détecté : {detected!r}")


================================================== PAGE 38 ==================================================
'LE PILOTAGE DE LA TRANSFORMATION NUMERIQUE DE L’ÉTAT PAR LA DIRECTION \nINTERMINISTERIELLE DU NUMERIQUE \n \n39 \nun département spécifique ; sa réalisation nécessite de relever de multiples défis tant les \ncompétences sont dispersées dans de nombreuses administrations et son pilotage à inventer.  \n1.3.'
→ Titre détecté : 'INTERMINISTERIELLE DU NUMERIQUE'

================================================== PAGE 39 ==================================================
'LE PILOTAGE DE LA TRANSFORMATION NUMERIQUE DE L’ÉTAT PAR LA DIRECTION \nINTERMINISTERIELLE DU NUMERIQUE \n \n40 \nont pu entretenir une confusion sur leur statut, facilitée par l’octroi d’adresses de courriel de la \ndirection, et se prévaloir d’une appartenance à la Dinum57.  \nLes effectifs du nouveau d'
→ Titre détecté : 'INTERMINISTERIELLE DU NUMERIQUE'

================================================== PAGE

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from rag_public_reports.config import DATA_DIR
from rag_public_reports.ingest import _detect_section_title, _get_patterns, _normalize

pdf_path = list((DATA_DIR / "raw").glob("*2024*ccomptes*"))[0]
pages = PyPDFLoader(str(pdf_path)).load()
patterns = _get_patterns("Cour des comptes")
title = "Le pilotage de la transformation numérique de l'État par la direction interministérielle du numérique"

for page in pages[38:44]:
    detected = _detect_section_title(page.page_content, patterns, title)
    print(f"Page {page.metadata.get('page')} → {detected!r}")

Page 38 → None
Page 39 → None
Page 40 → None
Page 41 → None
Page 42 → None
Page 43 → None


In [4]:
from rag_public_reports.ingest import _normalize

# Le titre tel qu'il est dans le catalogue
title = "Le pilotage de la transformation numérique de l'État par la direction interministérielle du numérique"

# La ligne détectée
line = "INTERMINISTERIELLE DU NUMERIQUE"

print(f"title normalisé : {_normalize(title)!r}")
print(f"line normalisée : {_normalize(line)!r}")
print(f"line in title ? : {_normalize(line) in _normalize(title)}")
print(f"title in line ? : {_normalize(title) in _normalize(line)}")

title normalisé : "le pilotage de la transformation numerique de l'etat par la direction interministerielle du numerique"
line normalisée : 'interministerielle du numerique'
line in title ? : True
title in line ? : False


In [3]:
import inspect
from rag_public_reports.ingest import _detect_section_title
print(inspect.getsource(_detect_section_title))

def _detect_section_title(
    text: str,
    compiled_patterns: list,
    title: str = "",
) -> str | None:
    """
    Parcourt toutes les lignes non vides du texte pour détecter un titre.
    Retourne le PREMIER titre trouvé (tronqué à 120 chars), ou None.

    Garde-fous appliqués pour éviter les faux positifs :
    - 1 mot de 5 chars ou moins → acronyme ou bruit (ex: "FTAP")
    - Ligne qui finit par un numéro isolé → header de page (ex: "DINUM   41")
    - Ligne répétée dans le texte de la page → header imprimé en en-tête
    - Ligne qui contient le titre du rapport → header récurrent inter-pages
    """
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    for line in lines:
        for pattern in compiled_patterns:
            if pattern.match(line):
                # Garde-fou 1 : acronyme trop court
                if len(line.split()) == 1 and len(line) <= 5:
                    continue
                # Garde-fou 2 : finit par un numéro de page
               

In [1]:
from rag_public_reports.ingest import _normalize, _get_patterns
from langchain_community.document_loaders import PyPDFLoader
from rag_public_reports.config import DATA_DIR
import re

title = "Le pilotage de la transformation numérique de l'État par la direction interministérielle du numérique"
patterns = _get_patterns("Cour des comptes")
pdf_path = list((DATA_DIR / "raw").glob("*2024*ccomptes*"))[0]
pages = PyPDFLoader(str(pdf_path)).load()

# On inspecte page 40 uniquement
page = pages[40]
lines = [l.strip() for l in page.page_content.split("\n") if l.strip()]

for line in lines[:6]:  # les 6 premières lignes suffisent
    print(f"\nLigne : {line!r}")
    for pattern in patterns:
        if pattern.match(line):
            print(f"  → Pattern matché : {pattern.pattern[:60]}")
            print(f"  → Garde-fou 4 : _normalize(line) = {_normalize(line)!r}")
            print(f"  → in title ?   : {_normalize(line) in _normalize(title)}")


Ligne : 'LE PILOTAGE DE LA TRANSFORMATION NUMERIQUE DE L’ÉTAT PAR LA DIRECTION'

Ligne : 'INTERMINISTERIELLE DU NUMERIQUE'
  → Pattern matché : ^(?:[A-ZÀÂÉÈÊËÎÏÔÙÛÜ]+[\s\-–:]+){2,}[A-ZÀÂÉÈÊËÎÏÔÙÛÜ]{2,}$
  → Garde-fou 4 : _normalize(line) = 'interministerielle du numerique'
  → in title ?   : True

Ligne : '41'

Ligne : 'Dinum de se doter de compétences de gestion statutaire des ressources humaines dont elle est'

Ligne : 'aujourd’hui dépourvue.'

Ligne : 'La constitution d’un corps fournirait toutefois un levier supplémentaire d’attractivité par'


In [2]:
# Vérifie sur l'ensemble du rapport que les vrais titres remontent
for page in pages:
    detected = _detect_section_title(page.page_content, patterns, title)
    if detected:
        print(f"Page {page.metadata.get('page'):3} → {detected!r}")

NameError: name '_detect_section_title' is not defined

In [4]:
for page_num in [0, 15, 28, 136]:
    page = pages[page_num]
    print(f"\n{'='*50} PAGE {page_num} {'='*50}")
    print(repr(page.page_content[:400]))


================================================== PAGE 0 ==================================================
'13, rue Cambon \uf06e 75100 PARIS CEDEX 01 \uf06e T +33 1 42 98 95 00 \uf06e\uf020www.ccomptes.fr \nPREMIÈRE CHAMBRE \nQUATRIÈME SECTION \n \nS-2024-0754 \nOBSERVATIONS DÉFINITIVES \n(Article R. 143-11 du code des juridictions financières) \nLE PILOTAGE DE LA \nTRANSFORMATION NUMERIQUE \nDE L’ÉTAT PAR LA DIRECTION \nINTERMINISTERIELLE DU \nNUMERIQUE \n \nExercices 2019-2023 \n \n \nLe présent document, qui a fait l’objet d’une contr'

================================================== PAGE 15 ==================================================
'LE PILOTAGE DE LA TRANSFORMATION NUMERIQUE DE L’ÉTAT PAR LA DIRECTION \nINTERMINISTERIELLE DU NUMERIQUE \n \n16 \n1 LA DINUM, UNE DIRECTION INTERMINISTERIELLE QUI \nPEINE A SE STRUCTURER FACE A DES STRATEGIES \nCHANGEANTES \nIssue des précédentes directions interministérielles chargées de la transformation \nnumérique de l’État, la Dinum

In [7]:
for page in pages:
    detected = _detect_section_title(page.page_content, patterns, title)
    if detected:
        print(f"Page {page.metadata.get('page'):3} → {detected!r}")

Page  15 → 'PEINE A SE STRUCTURER FACE A DES STRATEGIES'
Page  28 → 'AE CP AE CP AE CP AE CP AE CP'
Page  29 → 'AE CP AE CP AE CP AE CP AE CP'
Page  79 → 'NOUVEAUX ENJEUX DU NUMERIQUE PUBLIC'


In [8]:
from rag_public_reports.rag import answer

reponse = answer(
    "Quelles sont les recommandations sur la gestion des ressources humaines ?",
    verbose=True
)
print(reponse)

🗄️  Vector store chargé — 2781 chunks en base
🔍 15 chunks récupérés

CHUNKS RÉCUPÉRÉS :

[1] Cour des comptes 2024 — INTERMINISTERIELLE DU NUMERIQUE
INTERMINISTERIELLE DU NUMERIQUE 
 
41 
Dinum de se doter de compétences de gestion statutaire des ressources humaines dont elle est 
aujourd’hui dépourvue.  
La constitution d’un corps fournirait toutefois un levier supplémentaire d’attractivité par 
les salaires, en offrant un cadre spécifique pour…

[2] Cour des comptes 2024 — INTERMINISTERIELLE DU NUMERIQUE
INTERMINISTERIELLE DU NUMERIQUE 
 
38 
ressources humaines. À cet égard, la feuille de route de la direction prévoit qu’elle pourra 
procéder, en lien avec la DSAF, à des expérimentations visant à réduire les délais de 
recrutement à la Dinum. Une expérimentation du recours à des contrats de projet …

[3] IGF 2023 — 2.1.5. Établir une trajectoire RH annuelle synthétisée par la DINUM
2.1.5. Établir une trajectoire RH annuelle synthétisée par la DINUM  
Alors que la pyramide des âges e

In [9]:
questions = [
    # Question précise sur un rapport
    "Quelles sont les recommandations de la Cour des comptes sur le budget de la DINUM ?",
    # Question transversale multi-rapports
    "Comment l'IGF et la Cour des comptes se rejoignent-ils sur la gouvernance numérique ?",
    # Question sur un rapport moins représenté
    "Quelles sont les conclusions du rapport CDC 2025 ?",
]

for q in questions:
    print(f"\n{'='*60}\n{q}\n{'='*60}")
    print(answer(q, k=10))


Quelles sont les recommandations de la Cour des comptes sur le budget de la DINUM ?
🗄️  Vector store chargé — 2781 chunks en base
🔍 10 chunks récupérés
## Recommandations budgétaires de la Cour des comptes sur la DINUM (2024)

Le rapport de la Cour des comptes de 2024 sur « Le pilotage de la transformation numérique par la DINUM » formule plusieurs recommandations à caractère budgétaire :

---

### 1. Simplification de l'architecture budgétaire

**Recommandation n° 2** *(DINUM, direction du budget)* :
> Rattacher les actions portées par le **programme 352 – Innovation et transformation numériques** au **programme 129 – Coordination du travail gouvernemental**.

Cette recommandation vise à remédier à l'éclatement des crédits entre plusieurs programmes (129, 349, 352, 363), source de complexité et de sous-exécution. La Cour souligne que la DINUM est organisée autour de **deux RFFIM et trois responsables de programmes**, ce qui nuit à la lisibilité de l'action publique.

---

### 2. Pouv